# 01 — Chia tách Dataset

## Ràng buộc buổi đầu thầy yêu cầu
- **Không sử dụng thư viện scikit-learn** cho phần cài đặt thuật toán chính.
- **Tập test thực tế luôn là tập thô** — không được tiền xử lý hay tham gia vào quá trình fit thông số.
- **Sử dụng các kỹ thuật chống rò rỉ dữ liệu (data leakage prevention)** trong toàn bộ pipeline.

## Giải pháp của nhóm đề ra

| Ràng buộc | Giải pháp |
| :--- | :--- |
| Không dùng scikit-learn | Chỉ sử dụng `scikit-learn` để **so sánh kết quả** . Toàn bộ thuật toán, pipeline tiền xử lý được cài đặt thuần NumPy/Pandas. |
| Tập test thực tế là tập thô | **Split tập thô đầu tiên** (Notebook 01) để chia 2 tập train/test ở dạng thô. Tập test được giữ độc lập hoàn toàn. Nếu cần tập val ở Notebook 04/05 thì split val từ train. |
| Chống rò rỉ dữ liệu | **Chỉ fit tham số duy nhất trên tập train**, xuất ra file `.pkl` để có thể tái sử dụng biến đổi dữ liệu mới trong tương lai một cách nhất quán. Val và test tuyệt đối không tham gia |

---

**Mục tiêu:**
Tải dataset thô, làm sạch dữ liệu (chuẩn hóa nhãn + giá trị phi lý),
kiểm tra chất lượng (TotalCharges lỗi, duplicate), thực hiện **stratified split**
80 / 20 (train / test) theo cột `Churn`, rồi lưu ra `data/processed/`.
Nếu cần tập **val**, sẽ được tách từ **train** tại Notebook 04/05.

**Thứ tự xử lý:**
1. Import / Setup
2. Import hàm làm sạch từ src/preprocessing.py
3. Đọc dữ liệu thô
4. **Làm sạch dữ liệu** (clean_target_column + clean_raw_data)
5. Kiểm tra TotalCharges
6. Kiểm tra duplicate
7. Xử lý duplicate hoàn toàn
8. **Xử lý customerID trùng nhưng dữ liệu khác nhau**
9. Định nghĩa hàm stratified_split()
10. Gọi stratified_split()
11. Kiểm tra tỷ lệ class Churn
12. Lưu file CSV
13. Tổng kết (Summary)

## 1 — Import thư viện

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)

print('Thư viện đã sẵn sàng.')
print(f'pandas version : {pd.__version__}')
print(f'numpy  version : {np.__version__}')

Thư viện đã sẵn sàng.
pandas version : 3.0.5
numpy  version : 2.5.3


## 2 — Import hàm làm sạch từ src/preprocessing.py

3 hàm này đã được viết sẵn trong `src/preprocessing.py` để tái sử dụng:
- `clean_target_column()`: chuẩn hóa nhãn Churn (Yes/No/1/False/NO...) về đúng 2 giá trị.
- `clean_raw_data()`: chuẩn hóa nhãn categorical (gender/SeniorCitizen/Contract/...),
  xử lý giá trị số phi lý (tenure âm/> 100, MonthlyCharges âm/>150), xóa duplicate hoàn toàn.
- `resolve_duplicate_customer_ids()`: xử lý customerID trùng nhưng dữ liệu khác nhau, ưu tiên
  giữ bản ghi ít NaN hơn — dùng chung được cho các bước sau nếu cần xử lý lại dữ liệu thô tương tự.

Dùng lại ở đây thay vì viết trùng logic.

In [2]:
import sys
sys.path.insert(0, os.path.join('..', 'src'))

from preprocessing import clean_target_column, clean_raw_data, resolve_duplicate_customer_ids
print('Import preprocessing functions: OK')

Import preprocessing functions: OK


## 3 — Đọc dữ liệu thô

Đọc file CSV gốc từ `data/raw/`.
In **shape**, **danh sách cột** và **5 dòng đầu** để nắm sơ bộ.

In [3]:
RAW_DATA_PATH = '../data/raw/Telco_Customer_Churn.csv'

df = pd.read_csv(RAW_DATA_PATH)

print(f'Shape (số dòng x số cột): {df.shape}')
print(f'\nDanh sách {df.shape[1]} cột:')
for i, col in enumerate(df.columns, 1):
    print(f'  [{i:02d}] {col:<30s}  dtype: {df[col].dtype}')

print('\n5 dòng đầu:')
df.head()

Shape (số dòng x số cột): (7143, 21)

Danh sách 21 cột:
  [01] customerID                      dtype: str
  [02] gender                          dtype: str
  [03] SeniorCitizen                   dtype: str
  [04] Partner                         dtype: str
  [05] Dependents                      dtype: str
  [06] tenure                          dtype: float64
  [07] PhoneService                    dtype: str
  [08] MultipleLines                   dtype: str
  [09] InternetService                 dtype: str
  [10] OnlineSecurity                  dtype: str
  [11] OnlineBackup                    dtype: str
  [12] DeviceProtection                dtype: str
  [13] TechSupport                     dtype: str
  [14] StreamingTV                     dtype: str
  [15] StreamingMovies                 dtype: str
  [16] Contract                        dtype: str
  [17] PaperlessBilling                dtype: str
  [18] PaymentMethod                   dtype: str
  [19] MonthlyCharges                  d

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,8272-ONJLV,Male,0,No,No,12.0,Yes,No,Fiber optic,No,...,Yes,No,Yes,Yes,One year,No,Electronic check,95.70,1184,No
1,6682-QJDGB,Male,0,No,Yes,40.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,55.25,2139.2,Yes
2,4250-WAROZ,Male,1,Yes,Yes,60.0,Yes,Yes,Fiber optic,Yes,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,93.25,5774.55,No
3,1073-XXCZD,Male,0,Yes,No,55.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),25.70,1443.65,No
4,8035-PWSEV,Female,0,No,No,6.0,Yes,No,Fiber optic,No,...,Yes,Yes,No,Yes,Month-to-month,Yes,Electronic check,89.25,487.05,No


## 4 — Làm sạch dữ liệu (chuẩn hóa nhãn + giá trị phi lý)

**Lý do phải chạy trước mọi thứ khác:**
Nếu Churn còn bẩn (9 biến thể: Yes, No, ' No', 1, False, NO, YES, no, N) thì:
- `groupby('Churn')` trong stratified split sẽ chia theo 9 group thay vì 2.
- 6-7 dòng có nhãn lạ sẽ bị dồn gần hết vào test set mà không được log ra.
- Duplicate check cũng sai vì 'No' != ' No'.

`clean_raw_data()` xử lý thêm: MonthlyCharges > 150 sẽ bị set NaN (giá trị lỗi
như 99999.99 phát hiện khi EDA). Nhờ đó bước xử lý customerID trùng ở mục 8
chỉ cần lọc theo NaN thay vì tự viết lại rule ngưỡng riêng.

`drop_customer_id=False` vì notebook 01 vẫn cần `customerID` để xử lý duplicate.

In [4]:
print('Trước khi làm sạch -- phân phối Churn gốc:')
print(df['Churn'].value_counts(dropna=False))

df = clean_target_column(df, target_col='Churn')
df = clean_raw_data(df, drop_customer_id=False)

print('\nSau khi làm sạch -- phân phối Churn:')
print(df['Churn'].value_counts(dropna=False))

Trước khi làm sạch -- phân phối Churn gốc:
Churn
No       5235
Yes      1900
 No         2
1           1
False       1
NO          1
YES         1
no          1
N           1
Name: count, dtype: int64
[clean_target_column] Cột 'Churn' — giá trị gốc:
    'No'                :  5,237
    'Yes'               :  1,900
    '1'                 :      1
    'False'             :      1
    'NO'                :      1
    'YES'               :      1
    'no'                :      1
    'N'                 :      1

  -> Map thành công: chỉ còn 'Yes' / 'No'.

  Giá trị sau khi map:
    'No'        :  5,241  (73.37%)
    'Yes'       :  1,902  (26.63%)
[clean_raw_data] Bước 0: Phát hiện 92 customerID bị trùng:
  0016-QLJIS: 2 dòng
  0078-XZMHT: 2 dòng
  0080-EMYVY: 2 dòng
  0080-OROZO: 2 dòng
  0148-DCDOS: 2 dòng
  0369-ZGOVK: 2 dòng
  0484-JPBRU: 2 dòng
  0562-KBDVM: 2 dòng
  0953-LGOVU: 2 dòng
  1024-GUALD: 2 dòng
  1074-AMIOH: 2 dòng
  1112-CUNAO: 2 dòng
  1173-NOEYG: 2 dòng
  1324-NLTJE: 2 

## 5 — Kiểm tra lỗi cột `TotalCharges`

Cột `TotalCharges` đã là `float64` sau bước làm sạch.
Kiểm tra lại số dòng NaN (với tenure=0, TotalCharges đã được fill = 0 rồi).

In [5]:
print(f"Kiểu dữ liệu 'TotalCharges' sau clean: {df['TotalCharges'].dtype}")

n_invalid = df['TotalCharges'].isna().sum()
print(f'Số dòng TotalCharges = NaN: {n_invalid}')

if n_invalid > 0:
    mask_invalid = df['TotalCharges'].isna()
    cols_show = [c for c in ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']
                 if c in df.columns]
    print(df.loc[mask_invalid, cols_show].to_string())
else:
    print('-> TotalCharges sạch, không có NaN sau clean.')

Kiểu dữ liệu 'TotalCharges' sau clean: float64
Số dòng TotalCharges = NaN: 0
-> TotalCharges sạch, không có NaN sau clean.


## 6 — Kiểm tra duplicate

Có 2 loại duplicate cần phân biệt:
- **Duplicate hoàn toàn**: toàn bộ các cột giống nhau → sẽ drop.
- **Duplicate theo customerID**: cùng 1 khách nhưng data khác nhau → xử lý riêng ở mục 8.

Lưu ý: `clean_raw_data()` đã log customerID trùng bên trong (Bước 0).
Cell này kiểm tra lại để đảm bảo số liệu nhất quán.

In [6]:
n_full_dup = df.duplicated().sum()
print(f'Số dòng duplicate hoàn toàn: {n_full_dup} dòng')

if 'customerID' in df.columns:
    n_id_dup = int(df.duplicated(subset=['customerID']).sum())
    print(f'Số dòng duplicate theo customerID: {n_id_dup} dòng')
    if n_id_dup > 0:
        dup_ids = df[df.duplicated(subset=['customerID'], keep=False)]['customerID'].unique()
        print(f'\ncustomerID bị trùng ({len(dup_ids)} ID):')
        print(df[df['customerID'].isin(dup_ids)]
              [['customerID', 'tenure', 'MonthlyCharges', 'Churn']]
              .sort_values('customerID').to_string())
else:
    n_id_dup = 0
    print('(Đã drop customerID, không kiểm tra duplicate theo ID.)')

print(f'\n-> Tổng kết: {n_full_dup} dòng duplicate hoàn toàn cần loại bỏ trước split.')

Số dòng duplicate hoàn toàn: 47 dòng
Số dòng duplicate theo customerID: 77 dòng

customerID bị trùng (77 ID):
      customerID  tenure  MonthlyCharges Churn
4284  0016-QLJIS    65.0           90.45    No
4397  0016-QLJIS    65.0           90.45    No
818   0080-EMYVY    14.0             NaN    No
4026  0080-EMYVY    14.0           51.45    No
5549  0080-OROZO    35.0           99.25    No
1030  0080-OROZO    35.0           99.25    No
3405  0148-DCDOS    25.0           94.70   Yes
6359  0148-DCDOS     NaN           94.70   Yes
2649  0369-ZGOVK    28.0           70.40    No
1758  0369-ZGOVK    28.0           70.40    No
6380  0484-JPBRU    41.0           25.25    No
573   0484-JPBRU    41.0           25.25    No
4365  0953-LGOVU    12.0           35.50    No
4309  0953-LGOVU    12.0           35.50    No
4377  1024-GUALD     1.0           24.80   Yes
1310  1024-GUALD     1.0           24.80   Yes
1443  1074-AMIOH    53.0          108.25    No
4675  1074-AMIOH    53.0          108.25    

## 7 — Xử lý duplicate hoàn toàn

Drop các dòng duplicate **trước khi split** — đây là bước bắt buộc.

**Lý do phải xử lý TRƯỚC split:** nếu giữ duplicate rồi mới split, 2 bản ghi trùng nhau
có thể rơi vào train và test cùng lúc → model học được thông tin từ test (leakage).

**Chỉ drop duplicate hoàn toàn** — các dòng cùng customerID nhưng data khác nhau
sẽ được xử lý riêng biệt ở mục 8 bằng logic chọn bản ghi hợp lệ hơn.

Lưu ý: bước làm sạch nhãn ở mục 4 KHÔNG tự tạo thêm hay xóa duplicate hoàn toàn
(chỉ chuẩn hóa nhãn); số liệu n_full_dup ở đây phản ánh đúng thực tế.

In [7]:
n_before = len(df)
print(f'Số dòng TRƯỚC khi loại duplicate hoàn toàn: {n_before}')

df = df.drop_duplicates(keep='first').reset_index(drop=True)

n_after   = len(df)
n_dropped = n_before - n_after
print(f'Số dòng SAU  khi loại duplicate hoàn toàn: {n_after}')
print(f'Số dòng đã loại bỏ                        : {n_dropped} dòng')

Số dòng TRƯỚC khi loại duplicate hoàn toàn: 7125
Số dòng SAU  khi loại duplicate hoàn toàn: 7078
Số dòng đã loại bỏ                        : 47 dòng


## 8 — Xử lý customerID trùng nhưng dữ liệu khác nhau

Logic xử lý đã được chuyển vào `src/preprocessing.py` (`resolve_duplicate_customer_ids`)
để dùng lại được cho các bước sau nếu cần, tránh viết trùng code.

In [8]:
n_before_id_fix = len(df)

df = resolve_duplicate_customer_ids(df)

n_after_id_fix   = len(df)
n_dropped_id_fix = n_before_id_fix - n_after_id_fix
print(f'Đã giải quyết customerID trùng: loại bỏ {n_dropped_id_fix} dòng')
print(f'Shape sau xử lý: {df.shape}')

# Drop customerID sau khi đã xử lý xong duplicate theo ID
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
    print('Đã drop cột customerID (không dùng làm feature).')

print('\nPhân phối Churn sau toàn bộ xử lý duplicate:')
churn_dist = df['Churn'].value_counts()
for label in churn_dist.index:
    pct = churn_dist[label] / len(df) * 100
    print(f'  Churn={label}: {churn_dist[label]:5d} dòng  ({pct:.2f}%)')

[resolve_duplicate_customer_ids] Đã loại 30 dòng trùng customerID (giữ bản ghi ít NaN nhất trong ['tenure', 'MonthlyCharges', 'TotalCharges']; hoà -> giữ bản ghi xuất hiện trước trong df).
Đã giải quyết customerID trùng: loại bỏ 30 dòng
Shape sau xử lý: (7048, 21)
Đã drop cột customerID (không dùng làm feature).

Phân phối Churn sau toàn bộ xử lý duplicate:
  Churn=No:  5178 dòng  (73.47%)
  Churn=Yes:  1870 dòng  (26.53%)


## 9 — Định nghĩa hàm `stratified_split()`

Hàm split thủ công, **không dùng `train_test_split` của scikit-learn**.

Nguyên tắc: với mỗi class trong `target_col`, shuffle theo seed cố định,
cắt theo tỷ lệ train/test, gộp lại — đảm bảo tỷ lệ class đồng đều trong 2 tập.

**Lưu ý:** Nếu cần tập **val** ở Notebook 04/05, hãy gọi lại hàm này trên `train_df` để tách val từ train.

In [9]:
def stratified_split(df, target_col, train_ratio=0.80,
                     test_ratio=0.20, random_state=42):
    """
    Chia DataFrame thành 2 tập (train / test) giữ nguyên tỷ lệ class.
    Không dùng sklearn.

    Nếu cần tập val, hãy gọi lại hàm này trên train_df sau khi đã split.
    """
    assert abs(train_ratio + test_ratio - 1.0) < 1e-9

    rng    = np.random.default_rng(random_state)
    trains, tests = [], []

    for class_val in df[target_col].unique():
        subset = df[df[target_col] == class_val].copy()
        idx    = rng.permutation(len(subset))
        subset = subset.iloc[idx].reset_index(drop=True)
        n       = len(subset)
        n_train = int(n * train_ratio)
        trains.append(subset.iloc[:n_train])
        tests.append( subset.iloc[n_train:])

    train_df = pd.concat(trains).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df  = pd.concat(tests) .sample(frac=1, random_state=random_state).reset_index(drop=True)
    return train_df, test_df

print('Hàm stratified_split() đã được định nghĩa.')

Hàm stratified_split() đã được định nghĩa.


## 10 — Gọi hàm `stratified_split()`

Thực hiện split với tỷ lệ **80 / 20** (train / test) và `random_state=42`.

Tập test được giữ **độc lập hoàn toàn** — không tham gia fit bất kỳ tham số nào.
Nếu cần val ở Notebook 04/05, gọi lại `stratified_split()` trên `train_df`.

In [10]:
TRAIN_RATIO  = 0.80
TEST_RATIO   = 0.20
RANDOM_STATE = 42

train_df, test_df = stratified_split(
    df, target_col='Churn',
    train_ratio=TRAIN_RATIO, test_ratio=TEST_RATIO,
    random_state=RANDOM_STATE
)

print('Split hoàn tất!')
print(f'  train : {len(train_df):5d} dòng  ({len(train_df)/len(df)*100:.1f}%)')
print(f'  test  : {len(test_df):5d} dòng  ({len(test_df)/len(df)*100:.1f}%)')
print(f'  {chr(45)*43}')
print(f'  Tổng  : {len(train_df)+len(test_df):5d} dòng  (sau dedup: {len(df)} dòng)')

# Xác nhận Churn chỉ còn Yes/No trong cả 2 tập
assert set(train_df['Churn'].unique()) <= {'Yes', 'No'}, 'FAIL: train có nhãn lạ'
assert set(test_df['Churn'].unique())  <= {'Yes', 'No'}, 'FAIL: test có nhãn lạ'
print('\nASSERT: cả 2 tập chỉ còn đúng 2 giá trị Yes/No trong Churn: PASS')

Split hoàn tất!
  train :  5638 dòng  (80.0%)
  test  :  1410 dòng  (20.0%)
  -------------------------------------------
  Tổng  :  7048 dòng  (sau dedup: 7048 dòng)

ASSERT: cả 2 tập chỉ còn đúng 2 giá trị Yes/No trong Churn: PASS


## 11 — Kiểm tra tỷ lệ class Churn trong từng tập

Sanity check: tỷ lệ Churn=Yes/No trong 2 tập phải xấp xỉ nhau và xấp xỉ FULL.

In [11]:
def print_churn_distribution(name, dataframe):
    counts = dataframe['Churn'].value_counts()
    pcts   = dataframe['Churn'].value_counts(normalize=True) * 100
    total  = len(dataframe)
    bar    = chr(9472) * 50
    print(bar)
    print(f'  Tập: {name:<10s} | Tổng: {total:,} dòng')
    print(bar)
    for label in ['No', 'Yes']:
        if label in counts:
            print(f'  Churn={label}: {counts[label]:5,d} dòng  ({pcts[label]:.2f}%)')
    print()

print('=' * 50)
print('  PHÂN PHỐI CLASS CHURN SAU KHI SPLIT')
print('=' * 50)
print()
print_churn_distribution('FULL (gốc)', df)
print_churn_distribution('TRAIN',      train_df)
print_churn_distribution('TEST',       test_df)

print('--> Tỷ lệ Churn=Yes/No trong 2 tập xấp xỉ nhau và xấp xỉ FULL: Stratified split đúng.')

  PHÂN PHỐI CLASS CHURN SAU KHI SPLIT

──────────────────────────────────────────────────
  Tập: FULL (gốc) | Tổng: 7,048 dòng
──────────────────────────────────────────────────
  Churn=No: 5,178 dòng  (73.47%)
  Churn=Yes: 1,870 dòng  (26.53%)

──────────────────────────────────────────────────
  Tập: TRAIN      | Tổng: 5,638 dòng
──────────────────────────────────────────────────
  Churn=No: 4,142 dòng  (73.47%)
  Churn=Yes: 1,496 dòng  (26.53%)

──────────────────────────────────────────────────
  Tập: TEST       | Tổng: 1,410 dòng
──────────────────────────────────────────────────
  Churn=No: 1,036 dòng  (73.48%)
  Churn=Yes:   374 dòng  (26.52%)

--> Tỷ lệ Churn=Yes/No trong 2 tập xấp xỉ nhau và xấp xỉ FULL: Stratified split đúng.


## 12 — Lưu file CSV

Lưu 2 tập ra `data/processed/`. Cả 2 file đã làm sạch nhãn nhưng chưa encode feature
(chưa dùng Target Statistics — sẽ xử lý ở notebook 03).

**Lưu ý chống data leakage:** Tập `test_raw.csv` sẽ **không** tham gia fit bất kỳ
tham số nào. Nếu cần tập val, Notebook 04/05 sẽ tự split từ `train_raw.csv`.

In [12]:
PROCESSED_DIR  = '../data/processed'
TRAIN_RAW_PATH = f'{PROCESSED_DIR}/train_raw.csv'
TEST_RAW_PATH  = f'{PROCESSED_DIR}/test_raw.csv'

train_df.to_csv(TRAIN_RAW_PATH, index=False)
test_df .to_csv(TEST_RAW_PATH,  index=False)

print('Đã lưu:')
for name, path, data in [
    ('train_raw', TRAIN_RAW_PATH, train_df),
    ('test_raw',  TEST_RAW_PATH,  test_df),
]:
    size_kb = os.path.getsize(path) / 1024
    print(f'  {name:<12}: {len(data):>6,} dòng   {path}  ({size_kb:.1f} KB)')

Đã lưu:
  train_raw   :  5,638 dòng   ../data/processed/train_raw.csv  (713.8 KB)
  test_raw    :  1,410 dòng   ../data/processed/test_raw.csv  (181.2 KB)


## 13 — Tổng kết (Summary)

Tổng kết toàn bộ những gì đã thực hiện trong notebook `01_split.ipynb`.

**Nguyên tắc chống data leakage:**
- Tập `test_raw.csv` độc lập hoàn toàn, không tham gia fit bất kỳ tham số nào.
- Mọi tham số (impute, encode...) chỉ được fit trên `train_raw.csv`.
- Val (nếu cần ở NB 04/05) sẽ được split từ `train_raw.csv`, không phải từ dữ liệu gốc.

In [13]:
import os

print('  TỔNG KẾT -- 01_split.ipynb')

print(f'\n Dataset gốc  : {RAW_DATA_PATH}')

print(f'\n Kiểm tra & xử lý chất lượng dữ liệu:')
_n_inv = df['TotalCharges'].isna().sum()
print(f'   Cột Churn/gender/SeniorCitizen          : đã chuẩn hóa nhãn (xem log mục 4)')
print(f'   MonthlyCharges > 150 (lỗi nhập liệu)    : đã set NaN trong clean_raw_data')
print(f'   TotalCharges NaN (sau clean)             : {_n_inv} dòng')
print(f'   Duplicate toàn bộ cột đã loại            : {n_dropped} dòng')
print(f'   Duplicate theo customerID (đã log)       : {n_id_dup} dòng')
print(f'   customerID trùng khác data đã xử lý      : {n_dropped_id_fix} dòng bị loại')

print(f'\n Kết quả split (80/20, stratify=Churn, random_state=42):')
print(f'   {chr(45)*60}')
print(f'   {"Tập":<12} {"Số dòng":>8}   {"File đã lưu"}')
print(f'   {chr(45)*60}')

for name, path, data in [
    ('train_raw', TRAIN_RAW_PATH, train_df),
    ('test_raw',  TEST_RAW_PATH,  test_df),
]:
    size_kb = os.path.getsize(path) / 1024
    print(f'   {name:<12} {len(data):>8,} dòng   {path}  ({size_kb:.1f} KB)')

print(f'   {chr(45)*60}')
print(f'   {"Tổng":<12} {len(train_df)+len(test_df):>8,} dòng')

print('\n Churn trong các tập:')
for name, data in [('TRAIN', train_df), ('TEST', test_df)]:
    yes_pct = (data['Churn'] == 'Yes').mean() * 100
    no_pct  = (data['Churn'] == 'No').mean() * 100
    print(f'   {name:<6}: Yes={yes_pct:.2f}%  No={no_pct:.2f}%')

print('\n Ghi chú: val (nếu cần) sẽ được split từ train tại NB 04/05.')
print('\n Bước tiếp theo: notebook 02_eda.ipynb (EDA trên train_raw.csv)')

  TỔNG KẾT -- 01_split.ipynb

 Dataset gốc  : ../data/raw/Telco_Customer_Churn.csv

 Kiểm tra & xử lý chất lượng dữ liệu:
   Cột Churn/gender/SeniorCitizen          : đã chuẩn hóa nhãn (xem log mục 4)
   MonthlyCharges > 150 (lỗi nhập liệu)    : đã set NaN trong clean_raw_data
   TotalCharges NaN (sau clean)             : 0 dòng
   Duplicate toàn bộ cột đã loại            : 47 dòng
   Duplicate theo customerID (đã log)       : 77 dòng
   customerID trùng khác data đã xử lý      : 30 dòng bị loại

 Kết quả split (80/20, stratify=Churn, random_state=42):
   ------------------------------------------------------------
   Tập           Số dòng   File đã lưu
   ------------------------------------------------------------
   train_raw       5,638 dòng   ../data/processed/train_raw.csv  (713.8 KB)
   test_raw        1,410 dòng   ../data/processed/test_raw.csv  (181.2 KB)
   ------------------------------------------------------------
   Tổng            7,048 dòng

 Churn trong các tập:
   TRA